<a href="https://colab.research.google.com/github/ekc2024/ScamGuard-MY/blob/main/credit-analyzer-complete/Credit_Analyzer_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ ScamGuard-MY: Financial Document Analyzer v2.0
### PDPA-Compliant AI-Powered Risk Assessment Pipeline

---

**What this does:**
1. Accepts PDF or spreadsheet (`.xlsx`/`.csv`) financial documents
2. Extracts text while **redacting all PII** (NRICs, emails, phones) for PDPA compliance
3. Auto-detects document type (Invoice / Purchase Order / Credit Report / Overdue Statement)
4. Extracts structured fields specific to each document type
5. Flags risk/exception lines using keyword scanning
6. Generates an AI-powered risk dashboard

---

### 📊 Pipeline Architecture

```
┌─────────────┐    ┌───────────────┐    ┌──────────────────┐    ┌───────────────────┐    ┌──────────────┐    ┌───────────────┐
│  INPUT FILE │───▶│  EXTRACT TEXT  │───▶│  MASK PII (PDPA) │───▶│  DETECT DOC TYPE  │───▶│  EXTRACT      │───▶│  DETECT RISKS  │
│ PDF/XLSX/CSV│    │ pdfplumber /   │    │  NRIC, Email,    │    │  invoice / PO /   │    │  STRUCTURED  │    │  keyword +    │
│             │    │ pandas         │    │  Phone → REDACTED│    │  credit / overdue │    │  FIELDS      │    │  pattern scan │
└─────────────┘    └───────────────┘    └──────────────────┘    └───────────────────┘    └──────────────┘    └───────────────┘
                                                                                                                      │
                                                                                                                      ▼
                                                                                                             ┌───────────────┐
                                                                                                             │  AI ANALYSIS  │
                                                                                                             │  + DASHBOARD  │
                                                                                                             └───────────────┘
```

### ✨ v2.0 Changelog
- **Fixed:** `NameError` from cells running out of order — all functions in one cell now
- **Fixed:** Spreadsheet extraction now handles merged cells and messy layouts
- **New:** `overdue_statement` document type (no longer mis-classified as invoice)
- **Improved:** Broader regex patterns support `YYYY-MM-DD` dates, `Total Payable`, etc.
- **Improved:** Expanded risk keywords (28 terms) catch more real-world red flags
- **Improved:** Dashboard with 3-tier colour scoring (green/amber/red)

---
## Step 1 — Install Dependencies
Run once per Colab session.

In [1]:
!pip install pdfplumber openpyxl reportlab -q
print("\u2705 Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 41.1 MB/s eta 0:00:00
✅ Dependencies installed.


---
## Step 2 — Core Engine (All Functions)

**Run this cell once.** Everything the pipeline needs is defined here — you can run all subsequent cells in any order without errors.

| Function | Purpose |
|----------|--------|
| `mask_experian_pii()` | Redacts NRIC, email, phone → PDPA compliance |
| `extract_from_spreadsheet()` | Reads `.xlsx`/`.csv` handling merged cells |
| `extract_any()` | Routes to PDF or spreadsheet extractor + masks PII |
| `detect_document_type()` | Classifies: overdue / PO / invoice / credit / unknown |
| `extract_structured_fields()` | Pulls type-specific key fields via regex |
| `detect_risks()` | Keyword scan for risk/exception lines |
| `analyze_document()` | Orchestrates full pipeline |
| `simulate_llm_analysis()` | Mock AI response (replace for production) |
| `render_financial_dashboard()` | HTML dashboard output |

In [2]:
import re
import os
import pdfplumber
import pandas as pd
from IPython.display import display, HTML


# ═══════════════════════════════════════════════════════════════════════
# 2.1  PII MASKING (PDPA Compliance Layer)
# ═══════════════════════════════════════════════════════════════════════
# Three regex patterns catch Malaysian PII formats:
#   - NRIC: 6 digits (DOB) + dash + 2 digits (state) + dash + 4 digits
#   - Email: standard RFC-like pattern
#   - Phone: Malaysian mobile starting with 01X
# Each is replaced with a placeholder BEFORE any analysis happens.

def mask_experian_pii(text: str) -> str:
    """Masks Malaysian NRICs, Emails, and Phone Numbers for PDPA Compliance."""
    nric_pattern = r"\b\d{6}-\d{2}-\d{4}\b"
    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
    phone_pattern = r"\b01[0-9][-\s]?\d{3,4}[-\s]?\d{4}\b"

    masked = re.sub(nric_pattern, "[REDACTED_NRIC]", text)
    masked = re.sub(email_pattern, "[REDACTED_EMAIL]", masked)
    masked = re.sub(phone_pattern, "[REDACTED_PHONE]", masked)
    return masked


# ═══════════════════════════════════════════════════════════════════════
# 2.2  TEXT EXTRACTION (PDF or Spreadsheet → raw text)
# ═══════════════════════════════════════════════════════════════════════
# Key fix: header=None prevents first row being swallowed as column names.
# NaN filtering handles merged cells gracefully.

def extract_from_spreadsheet(file_path: str) -> str:
    """Extracts spreadsheet data into text. Handles merged cells and messy layouts."""
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path, header=None)
    else:
        df = pd.read_excel(file_path, header=None)

    lines = []
    for _, row in df.iterrows():
        values = [str(v).strip() for v in row if pd.notna(v) and str(v).strip()]
        if values:
            lines.append(" | ".join(values))
    return "\n".join(lines)


def extract_any(file_path: str) -> str:
    """Routes file to correct extractor, then masks PII. Single entry point."""
    if file_path.lower().endswith((".xlsx", ".xls", ".csv")):
        raw_text = extract_from_spreadsheet(file_path)
    else:
        raw_text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    raw_text += text + "\n"
    return mask_experian_pii(raw_text)


# ═══════════════════════════════════════════════════════════════════════
# 2.3  DOCUMENT TYPE DETECTION
# ═══════════════════════════════════════════════════════════════════════
# Priority order: overdue > PO > invoice > credit > unknown
# Overdue statements often contain the word "invoice" but should NOT
# be classified as standard invoices.

def detect_document_type(sanitized_text: str) -> str:
    """Classifies document type using keyword signals, checked most-specific first."""
    text_lower = sanitized_text.lower()

    # Priority 1: Overdue / Collection / Billing Statements
    overdue_signals = ["overdue", "billing statement", "delinquency", "collection warning",
                       "past due", "outstanding balance", "default"]
    debt_signals = ["liabilities", "debt", "lawsuit", "legal action", "credit default"]
    has_overdue = any(s in text_lower for s in overdue_signals)
    has_debt = any(s in text_lower for s in debt_signals)
    if has_overdue and has_debt:
        return "overdue_statement"

    # Priority 2: Purchase Orders
    if ("purchase order" in text_lower or
        re.search(r"\bpo\s*number\b", text_lower) or
        re.search(r"\bpo[\s#\-]*\d", text_lower)):
        return "purchase_order"

    # Priority 3: Standard Tax Invoices
    if ("tax invoice" in text_lower or
        "invoice no" in text_lower or
        "invoice number" in text_lower or
        re.search(r"\binvoice\b", text_lower)):
        return "invoice"

    # Priority 4: Credit Reports / Assessments
    if ("credit assessment" in text_lower or
        "credit report" in text_lower or
        "experian" in text_lower):
        return "credit_report"

    return "unknown"


# ═══════════════════════════════════════════════════════════════════════
# 2.4  STRUCTURED FIELD EXTRACTION (per document type)
# ═══════════════════════════════════════════════════════════════════════
# Once we know the type, we apply type-specific regex rules to pull
# the key fields a finance team cares about.

def _first_match(pattern, text, group=1, flags=re.IGNORECASE):
    """Returns first regex match group, or None."""
    m = re.search(pattern, text, flags)
    return m.group(group).strip() if m else None


def extract_invoice_fields(sanitized_text: str) -> dict:
    """Extracts key fields from a standard tax invoice."""
    return {
        "invoice_no": _first_match(
            r"invoice\s*(?:no|number)\.?[:\s]+([A-Za-z0-9\-]+)", sanitized_text),
        "invoice_date": _first_match(
            r"invoice\s*date[:\s]+([\d/\-]+)", sanitized_text),
        "due_date": _first_match(
            r"due\s*date[:\s]+([\d/\-]+)", sanitized_text),
        "bill_to": _first_match(
            r"bill\s*to[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_due": _first_match(
            r"total\s*(?:due|payable|amount)[:\s]*(RM[\s\d,\.]+)", sanitized_text),
        "payment_status": _first_match(
            r"payment\s*status[:\s]+([A-Za-z ]+?)(?:\n|\||$)", sanitized_text),
    }


def extract_po_fields(sanitized_text: str) -> dict:
    """Extracts key fields from a purchase order."""
    return {
        "po_number": _first_match(
            r"po\s*number[:\s]+([A-Za-z0-9\-]+)", sanitized_text),
        "po_date": _first_match(
            r"po\s*date[:\s]+([\d/\-]+)", sanitized_text),
        "expected_delivery": _first_match(
            r"expected\s*delivery[:\s]+([\d/\-]+)", sanitized_text),
        "vendor": _first_match(
            r"vendor[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_po_value": _first_match(
            r"total\s*po\s*value[:\s]*(RM[\s\d,\.]+)", sanitized_text),
        "payment_terms": _first_match(
            r"payment\s*terms[:\s]+([A-Za-z0-9 ]+?)(?:\n|\||$)", sanitized_text),
    }


def extract_credit_report_fields(sanitized_text: str) -> dict:
    """Extracts key fields from a credit assessment report."""
    return {
        "company_name": _first_match(
            r"company\s*name[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "director_name": _first_match(
            r"director\s*(?:name|in[- ]charge)[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_liabilities": _first_match(
            r"(?:total\s*(?:outstanding\s*)?)?(?:liabilities|unpaid)[^\d]*(RM[\s\d,\.]+)",
            sanitized_text),
    }


def extract_overdue_statement_fields(sanitized_text: str) -> dict:
    """Extracts key fields from an overdue billing / collection statement."""
    return {
        "creditor": _first_match(
            r"(?:vendor\s*/\s*)?creditor[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "debtor": _first_match(
            r"(?:debtor\s*(?:company)?)[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_unpaid": _first_match(
            r"(?:total\s*(?:unpaid|outstanding)\s*(?:liabilities|amount)?)[^\d]*(RM[\s\d,\.]+)",
            sanitized_text),
        "risk_status": _first_match(
            r"(?:account\s*)?risk\s*status[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "payment_warning": _first_match(
            r"payment\s*warning[:\s]+(.+?)(?:\n|$)", sanitized_text),
    }


def extract_structured_fields(sanitized_text: str, doc_type: str) -> dict:
    """Dispatches to the right field extractor based on detected type."""
    extractors = {
        "invoice": extract_invoice_fields,
        "purchase_order": extract_po_fields,
        "credit_report": extract_credit_report_fields,
        "overdue_statement": extract_overdue_statement_fields,
    }
    extractor = extractors.get(doc_type)
    return extractor(sanitized_text) if extractor else {}


# ═══════════════════════════════════════════════════════════════════════
# 2.5  RISK / EXCEPTION DETECTION
# ═══════════════════════════════════════════════════════════════════════
# Fast deterministic keyword scan. The LLM layer adds semantic depth.

DEFAULT_RISK_KEYWORDS = [
    # Original keywords
    "lawsuit", "liabilities", "deteriorating", "unpaid", "debt", "risk",
    "overdue", "suspension", "default", "penalty", "dispute", "terminate",
    # Expanded v2 keywords
    "delinquent", "delinquency", "arrears", "collections", "legal action",
    "credit default", "write-off", "write off", "bad debt", "non-performing",
    "past due", "outstanding balance", "service suspension", "warning",
    "high risk", "severe", "forfeiture", "indemnity",
]


def detect_risks(sanitized_text: str, risk_keywords=None) -> list:
    """Flags lines containing risk-related keywords (deduplicated)."""
    if risk_keywords is None:
        risk_keywords = DEFAULT_RISK_KEYWORDS
    detected_risks = []
    for line in sanitized_text.split('\n'):
        if any(kw in line.lower() for kw in risk_keywords):
            if line.strip() and line.strip() not in detected_risks:
                detected_risks.append(line.strip())
    return detected_risks


# ═══════════════════════════════════════════════════════════════════════
# 2.6  FULL PIPELINE ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════

def analyze_document(file_path: str) -> dict:
    """Full pipeline: extract → mask PII → detect type → extract fields → detect risks."""
    print(f"\n{'='*70}")
    print(f"📄 Analyzing: {os.path.basename(file_path)}")
    print(f"{'='*70}")

    try:
        sanitized_text = extract_any(file_path)
    except FileNotFoundError:
        print(f"\u274c Error: Could not find '{file_path}'")
        print("   Please generate samples (Step 3) or upload a file (Step 4) first.")
        return {}
    except Exception as e:
        print(f"\u274c Error reading '{file_path}': {e}")
        return {}

    doc_type = detect_document_type(sanitized_text)
    fields = extract_structured_fields(sanitized_text, doc_type)
    risks = detect_risks(sanitized_text)

    result = {
        "file_path": file_path,
        "document_type": doc_type,
        "sanitized_text": sanitized_text,
        "fields": fields,
        "risks": risks,
    }

    # Pretty-print output
    print(f"\n\ud83c\udfaf Detected Document Type: {doc_type.upper().replace('_', ' ')}")

    print(f"\n\ud83d\udd12 PDPA SANITIZED TEXT:")
    print("-" * 50)
    print(sanitized_text[:2000])  # Cap display at 2000 chars
    if len(sanitized_text) > 2000:
        print(f"... [{len(sanitized_text) - 2000} more characters]")

    print(f"\n\ud83d\udccb STRUCTURED FIELDS:")
    print("-" * 50)
    if fields:
        for k, v in fields.items():
            status = v if v else '\u26a0\ufe0f (not found)'
            print(f"  {k:20s}: {status}")
    else:
        print("  (No field rules for this document type.)")

    print(f"\n\u26a1 RISK / EXCEPTION LINES ({len(risks)} found):")
    print("-" * 50)
    if risks:
        for idx, r in enumerate(risks, 1):
            print(f"  [{idx}] {r}")
    else:
        print("  \u2705 No risk keywords detected.")

    return result


# Backward compatibility
def analyze_credit_report(file_path: str):
    return analyze_document(file_path)


# ═══════════════════════════════════════════════════════════════════════
# 2.7  AI ANALYSIS + DASHBOARD
# ═══════════════════════════════════════════════════════════════════════

def simulate_llm_analysis(sanitized_text: str, doc_type: str = "credit_report", fields: dict = None) -> dict:
    """
    Mock LLM response for demo. Replace with real API call for production.
    See Step 7 for the real implementation template.
    """
    # Generate a dynamic risk score based on actual risk detection
    risks = detect_risks(sanitized_text)
    num_risks = len(risks)

    # Simple heuristic scoring based on detected risks
    if num_risks == 0:
        score = 15
        category = "Low Risk"
    elif num_risks <= 2:
        score = 35
        category = "Medium Risk"
    elif num_risks <= 4:
        score = 55
        category = "Medium-High Risk"
    else:
        score = 75
        category = "High Risk"

    # Adjust for overdue statements (inherently high risk)
    if doc_type == "overdue_statement":
        score = max(score, 65)
        category = "High Risk" if score >= 65 else category

    # Generate recommendations based on doc type
    recs_by_type = {
        "invoice": [
            "Verify payment status and follow up if overdue.",
            "Cross-check total amounts against purchase orders.",
            "Confirm delivery of goods/services before payment release.",
        ],
        "overdue_statement": [
            "Immediately verify debt legitimacy with internal records.",
            "Assess legal exposure and consult legal counsel if lawsuit mentioned.",
            "Prioritise settlement negotiation to avoid credit score damage.",
            "Check if debtor has disputed any of the claimed amounts.",
        ],
        "purchase_order": [
            "Verify vendor financial health before committing large orders.",
            "Ensure indemnity clauses are in place if risk notes present.",
            "Confirm delivery timelines align with project deadlines.",
        ],
        "credit_report": [
            "Review flagged financial distress indicators in detail.",
            "Request updated financials from the assessed entity.",
            "Consider credit limit reduction or additional guarantees.",
        ],
    }

    return {
        "risk_score": score,
        "risk_category": category,
        "executive_summary": (
            f"Document classified as {doc_type.replace('_', ' ')}. "
            f"{num_risks} risk indicator(s) detected. "
            f"{'Immediate attention recommended.' if score >= 50 else 'Standard review sufficient.'}"
        ),
        "actionable_recommendations": recs_by_type.get(doc_type, [
            "Review document manually for unrecognized format.",
            "Consult finance team for risk assessment.",
        ]),
        "explainability_flag": (
            f"Score based on {num_risks} detected risk line(s) and document type '{doc_type}'."
        ),
    }


def render_financial_dashboard(ai_insights: dict, doc_type: str = "", fields: dict = None):
    """Renders AI insights as an HTML dashboard."""
    score = ai_insights["risk_score"]
    score_color = "#dc2626" if score >= 50 else "#d97706" if score >= 30 else "#16a34a"
    fields = fields or {}

    not_found_html = '<i style="color:#9ca3af">not found</i>'
    fields_html = "".join(
        "<li><b>{}:</b> {}</li>".format(k.replace('_', ' ').title(), v if v else not_found_html)
        for k, v in fields.items()
    ) or "<li><i>No structured fields for this document type.</i></li>"

    # Type badge colour
    type_colors = {
        "invoice": "#2563eb", "purchase_order": "#7c3aed",
        "credit_report": "#dc2626", "overdue_statement": "#ea580c", "unknown": "#6b7280"
    }
    type_color = type_colors.get(doc_type, "#6b7280")
    type_label = doc_type.replace('_', ' ').title() if doc_type else 'Unknown'

    dashboard_html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                border: 2px solid #e5e7eb; border-radius: 12px; padding: 24px; max-width: 850px;
                background: linear-gradient(135deg, #fafafa 0%, #ffffff 100%);">

        <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #e5e7eb; padding-bottom:12px;">
            <h2 style="margin:0; color:#1f2937;">📊 ScamGuard-MY Risk Dashboard</h2>
            <span style="background:{type_color}; color:white; padding:4px 12px; border-radius:20px; font-size:13px; font-weight:600;">
                {type_label}
            </span>
        </div>

        <div style="display:flex; justify-content:space-between; margin-top:20px; gap:20px;">
            <div style="background:#f9fafb; padding:20px; border-radius:10px; width:40%; text-align:center; border:1px solid #e5e7eb;">
                <h3 style="margin:0 0 8px 0; color:#4b5563; font-size:14px; text-transform:uppercase; letter-spacing:1px;">Risk Score</h3>
                <h1 style="margin:0; font-size:56px; color:{score_color}; font-weight:800;">{score}</h1>
                <p style="margin:4px 0 0 0; font-size:12px; color:#6b7280;">out of 100</p>
                <p style="margin:8px 0 0 0; font-weight:700; color:{score_color}; font-size:15px;">{ai_insights['risk_category']}</p>
            </div>

            <div style="width:58%;">
                <h3 style="margin:0 0 8px 0; color:#4b5563; font-size:14px; text-transform:uppercase; letter-spacing:1px;">Executive Summary</h3>
                <p style="color:#374151; line-height:1.6; margin:0;">{ai_insights['executive_summary']}</p>
                <p style="font-size:11px; color:#9ca3af; margin-top:10px; font-style:italic;">
                    💡 {ai_insights['explainability_flag']}
                </p>
            </div>
        </div>

        <h3 style="color:#4b5563; margin-top:24px; font-size:14px; text-transform:uppercase; letter-spacing:1px;">
            📄 Extracted Fields
        </h3>
        <ul style="color:#374151; line-height:1.8; margin:8px 0; padding-left:20px;">
            {fields_html}
        </ul>

        <h3 style="color:#4b5563; margin-top:24px; font-size:14px; text-transform:uppercase; letter-spacing:1px;">
            \u26a1 Actionable Recommendations
        </h3>
        <ul style="color:#374151; line-height:1.8; margin:8px 0; padding-left:20px;">
            {"".join(f'<li>{item}</li>' for item in ai_insights['actionable_recommendations'])}
        </ul>

        <p style="font-size:11px; color:#9ca3af; margin-top:24px; border-top:1px solid #f3f4f6; padding-top:12px;">
            \ud83d\udd12 <b>PDPA Notice:</b> All NRICs, emails, and phone numbers were redacted before analysis.
            No personal data is stored, logged, or transmitted to any LLM.
        </p>
    </div>
    """
    display(HTML(dashboard_html))


# ═════ READY ═════
print("\u2705 Core Engine v2.0 loaded successfully!")
print("   All functions ready. Proceed to Step 3 or 4.")

✅ Core Engine v2.0 loaded successfully!
   All functions ready. Proceed to Step 3 or 4.


---
## Step 3 — (Option A) Generate Mock Sample Documents

Creates three simulated PDFs for testing: credit report, invoice, and purchase order.

**Skip this if uploading your own file in Step 4.**

In [3]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


def create_mock_credit_report(filename="Credit_Assessment_Pinnacle.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "EXPERIAN COMMERCIAL CREDIT ASSESSMENT")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 700, "Company Name: Pinnacle Tech Solutions")
    c.drawString(100, 680, "Director Name: Ahmad Razak")
    c.drawString(100, 660, "Director NRIC: 880412-14-5531")
    c.drawString(100, 640, "Contact Email: ahmad.razak@pinnacle.com.my")
    c.drawString(100, 620, "Mobile Phone: 012-3456789")
    c.drawString(100, 580, "FINANCIAL SUMMARY & RISK METRICS:")
    c.drawString(100, 560, "- The entity shows flags of deteriorating capital reserves.")
    c.drawString(100, 540, "- High risk exposure detected due to severe unpaid supplier invoices.")
    c.drawString(100, 520, "- Active lawsuit filed by major vendor on 15/04/2026.")
    c.drawString(100, 500, "- Total outstanding liabilities exceed RM 450,000.")
    c.save()
    print(f"\u2705 Created {filename}")


def create_mock_invoice(filename="Sample_Invoice_INV-2026-0451.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "TAX INVOICE")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 710, "Invoice No: INV-2026-0451")
    c.drawString(100, 690, "Invoice Date: 12/08/2026")
    c.drawString(100, 670, "Due Date: 26/08/2026")
    c.drawString(100, 630, "Bill To: Meridian Trading Sdn Bhd")
    c.drawString(100, 610, "Attn: Siti Nurhaliza binti Ismail")
    c.drawString(100, 590, "Contact Email: siti.ismail@meridiantrading.com.my")
    c.drawString(100, 570, "Contact Phone: 019-8827364")
    c.drawString(100, 550, "Company Reg NRIC (Director): 850627-08-5142")
    c.drawString(100, 510, "ITEMS:")
    c.drawString(100, 495, "1. Industrial Packaging Rolls x 500 units - RM 8,500.00")
    c.drawString(100, 480, "2. Freight & Logistics Fee            - RM 620.00")
    c.drawString(100, 465, "3. Handling Surcharge                 - RM 150.00")
    c.drawString(100, 430, "Subtotal: RM 9,270.00")
    c.drawString(100, 415, "SST (6%): RM 556.20")
    c.drawString(100, 400, "TOTAL DUE: RM 9,826.20")
    c.drawString(100, 365, "Payment Status: UNPAID")
    c.drawString(100, 350, "Note: This is the third overdue reminder. Outstanding balance")
    c.drawString(100, 335, "has exceeded 45 days past due date. Risk of service suspension")
    c.drawString(100, 320, "if payment is not received within 7 days.")
    c.save()
    print(f"\u2705 Created {filename}")


def create_mock_po(filename="Sample_PO_PO-8842.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "PURCHASE ORDER")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 710, "PO Number: PO-8842")
    c.drawString(100, 690, "PO Date: 05/08/2026")
    c.drawString(100, 670, "Expected Delivery: 20/08/2026")
    c.drawString(100, 630, "Vendor: Apex Steelworks Sdn Bhd")
    c.drawString(100, 610, "Vendor Contact: Ahmad Faiz bin Zulkifli")
    c.drawString(100, 590, "Vendor Email: faiz.zulkifli@apexsteel.com.my")
    c.drawString(100, 570, "Vendor Phone: 012-5563981")
    c.drawString(100, 550, "Vendor Director NRIC: 780315-10-6231")
    c.drawString(100, 510, "ORDERED ITEMS:")
    c.drawString(100, 495, "1. Galvanized Steel Sheets 2mm x 200 units - RM 24,000.00")
    c.drawString(100, 480, "2. Mounting Brackets x 400 units           - RM 3,200.00")
    c.drawString(100, 465, "3. Delivery & Installation Service         - RM 1,500.00")
    c.drawString(100, 430, "TOTAL PO VALUE: RM 28,700.00")
    c.drawString(100, 395, "Payment Terms: Net 30")
    c.drawString(100, 380, "Vendor Risk Note: Vendor has an active lawsuit filed by a")
    c.drawString(100, 365, "former subcontractor regarding unpaid labour claims. Approve")
    c.drawString(100, 350, "with caution and require signed indemnity before deposit release.")
    c.save()
    print(f"\u2705 Created {filename}")


# Generate all three mocks
create_mock_credit_report()
create_mock_invoice()
create_mock_po()

# Set default file for testing
sample_file_path = "Sample_Invoice_INV-2026-0451.pdf"
print(f"\n\ud83d\udc49 sample_file_path = '{sample_file_path}'")
print("   Change this variable or upload a file in Step 4 to test different documents.")

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 149-150: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/zmq/eventloop/zmqstream.py", line 551, in _run_callback
    f = callback(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 120, in _handle_event
    event_f()
    ~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
    ~~~~~~~~~~~~~~~~~^
        self.pub_thread,
        ^^^^^^^^^^^^^^^^
    ...<3 lines>...
        ident=self.topic,
 

---
## Step 4 — (Option B) Upload Your Own Document

Upload a real PDF, `.xlsx`, or `.csv` file. This will set `sample_file_path` to your uploaded file.

**If you already ran Step 3, this will override `sample_file_path`.**

In [ ]:
from google.colab import files

print("📤 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...")
uploaded = files.upload()

if uploaded:
    sample_file_path = list(uploaded.keys())[0]
    print(f"\n\u2705 '{sample_file_path}' uploaded successfully!")
    print(f"   File size: {len(list(uploaded.values())[0]):,} bytes")
else:
    print("\u26a0\ufe0f No file uploaded \u2014 keeping existing sample_file_path.")

print(f"\n\ud83d\udc49 Current sample_file_path = '{sample_file_path}'")

📤 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...


---
## Step 5 — Run Analysis Pipeline

Runs the full pipeline on `sample_file_path`:
1. Extract text (PDF or spreadsheet)
2. Redact PII (NRIC, email, phone)
3. Detect document type
4. Extract structured fields
5. Flag risk/exception lines

In [ ]:
result = analyze_document(sample_file_path)

---
## Step 6 — Generate AI Dashboard

Feeds the pipeline output into the AI analysis engine and renders the risk dashboard.

> ⚠️ The current `simulate_llm_analysis()` uses a **heuristic scoring model** (not a real LLM). It varies based on the number of risks detected and document type. For production, replace with a real LLM call (see Step 7).

In [ ]:
if result:
    print("\n\ud83e\udd16 Running AI Risk Analysis...\n")
    ai_results = simulate_llm_analysis(
        result["sanitized_text"],
        doc_type=result["document_type"],
        fields=result["fields"],
    )
    render_financial_dashboard(
        ai_results,
        doc_type=result["document_type"],
        fields=result["fields"]
    )
else:
    print("\u26a0\ufe0f No result to analyze. Please run Step 3 or 4 first, then Step 5.")

---
## Step 6b — Quick One-Shot (Pipeline + Dashboard in one cell)

Convenience cell: runs both the extraction pipeline AND the dashboard in one go.

In [ ]:
# One-shot: analyze + dashboard
result = analyze_document(sample_file_path)

if result:
    print("\n\ud83e\udd16 Running AI Risk Analysis...\n")
    ai_results = simulate_llm_analysis(
        result["sanitized_text"],
        doc_type=result["document_type"],
        fields=result["fields"],
    )
    render_financial_dashboard(
        ai_results,
        doc_type=result["document_type"],
        fields=result["fields"]
    )

---
## Step 7 — (Production) Replace Mock with Real LLM

To make the AI analysis actually read and reason about your documents, replace `simulate_llm_analysis()` with a real API call.

**Option A: Anthropic Claude**
```python
!pip install anthropic -q
import anthropic, json
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

def simulate_llm_analysis(sanitized_text, doc_type="credit_report", fields=None):
    prompt = f'''Context: You are a corporate financial risk analyst.
    Document type: {doc_type}
    Structured fields: {fields}
    Task: Assess risk. Respond ONLY with JSON:
    {{"risk_score": int 1-100, "risk_category": str, "executive_summary": str,
      "actionable_recommendations": [str], "explainability_flag": str}}
    Data: {sanitized_text[:3000]}'''

    response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)
```

**Option B: Google Gemini**
```python
!pip install google-generativeai -q
import google.generativeai as genai, json
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-1.5-flash')

def simulate_llm_analysis(sanitized_text, doc_type="credit_report", fields=None):
    prompt = f'''...(same as above)...'''
    response = model.generate_content(prompt)
    raw = response.text.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)
```

Store API keys using Colab's Secrets manager (`userdata.get()`) — never hardcode them.